# Economy Sim

A notebook which launches an economy simulator.

In [ ]:
import uuid
from decimal import Decimal
import datetime
from pathlib import Path

from configs.config import GLOBAL_ECONOMY_LOG_DIR
from utils.logger import create_custom_logger, override_print_with_logger

from exceptions.not_enough_money_error import NotEnoughMoneyError

from models.enums import CompetitionModel
from models.config_model import WholesalerConfig, RetailerConfig, ConsumerConfig


import random

import pandas as pd
from pandas import DataFrame

available_lang_model_and_temperature_combinations = [
    {
        "model": "mistralai/ministral-3-3b",
        "temperature": 0.7
    },
    {
        "model": "mistralai/ministral-3-3b",
        "temperature": 0.3
    },
    {
        "model": "google/gemma-3-4b",
        "temperature": 0.7
    },
    {
        "model": "google/gemma-3-4b",
        "temperature": 0.3
    },
    # {
    #     "model": "google/gemma-4-26b-a4b",
    #     "temperature": 0.7
    # },
    # {
    #     "model": "google/gemma-4-26b-a4b",
    #     "temperature": 0.5
    # },
    # {
    #     "model": "qwen/qwen3.5-9b",
    #     "temperature": 0.7
    # },
    # {
    #     "model": "qwen/qwen3.5-9b",
    #     "temperature": 0.3
    # },
    {
        "model": "mistralai/ministral-3-14b-reasoning",
        "temperature": 0.7
    },
    {
        "model": "google/gemma-3-12b",
        "temperature": 0.7
    },
]

# available_language_models = [
#     "google/gemma-2-27b",
#     "mistralai/ministral-3-3b",
#     "google/gemma-3-4b",
#     # "google/gemma-3n-e4b",
#     "google/gemma-3-12b",
#     "mistralai/ministral-3-14b-reasoning"
# ]

available_retailer_names = [
    "Bobs Bargains",
    "Sally's Sales",
    "Tom's Treasures",
    "Alice's Emporium",
    "Eve's Essentials",
    "Charlie's Exquisite Goods",
    "The Corner Store",
    "Penny's Place",
    "Main Street Market",
    "The Trading Post",
    "Frank's Fine Finds",
    "The Daily Deal",
    "Harbor View Shop",
    "Riverside Retail",
    "The Marketplace",
    "Dixon & Sons",
    "Mulberry Lane",
    "The Exchange",
    "Greenfield General",
    "The Depot",
    "Hillside Merchants",
    "Patty's Picks",
    "The Storehouse",
    "The Good Buy",
    "The Open Door",
    "Cooper & Co.",
]

### Logging

This cell creates a logger, and attaches it to `print` so that 
I can just use `print(f'whatever')` without needing to 
remember to use custom_logger.info.

In [2]:
now = datetime.datetime.now()
log_path = Path(GLOBAL_ECONOMY_LOG_DIR) / f"{now.year:04d}" / f"{now.month:02d}" / f"{now.day:02d}"
    
economy_logger = create_custom_logger(log_path, logger_name='economy')
original_print = print
override_print_with_logger(economy_logger)
economy_logger.info('Starting economy simulation at {}'.format(now.strftime("%Y-%m-%d %H:%M:%S")))

print_test_statements = True
if print_test_statements:
    # These are test lines to verify that the logger is working correctly. 
    # Set print_test_statements to False to disable them.
    economy_logger.debug("TEST: This is a DEBUG message (only in notebook)")
    economy_logger.info("TEST: This is an INFO message (in both file and notebook)")
    economy_logger.error("TEST: This is an ERROR message (in both file and notebook)")
    print("TEST: This is a test print statement (should appear in both file and notebook)")

2026-06-02 11:10:55,910 - economy - INFO - Starting economy simulation at 2026-06-02 11:10:55
2026-06-02 11:10:55,911 - economy - DEBUG - TEST: This is a DEBUG message (only in notebook)
2026-06-02 11:10:55,912 - economy - INFO - TEST: This is an INFO message (in both file and notebook)
2026-06-02 11:10:55,912 - economy - ERROR - TEST: This is an ERROR message (in both file and notebook)
2026-06-02 11:10:55,912 - economy - INFO - TEST: This is a test print statement (should appear in both file and notebook)


### Actor Objects

This cell contains objects for the different types of actors in the economy.

In [3]:

class Actor:
    name: str
    id: uuid
    language_model_name: str | None
    language_model_temperature: Decimal | None

    def __init__(self, name: str, language_model_name: str | None = None, langage_model_temperature: Decimal | None = None):
        self.name = name
        self.id = uuid.uuid4()
        self.language_model_name = language_model_name
        self.language_model_temperature = langage_model_temperature
       
       
class BankAccount:
    balance: Decimal
    owner: str
    owner_type: str

    def __init__(self, initial_balance: Decimal = Decimal(0)):
        self.balance = initial_balance



class TransactionLogEntry:
    timestamp: datetime.datetime
    sender: uuid
    sender_type: str
    receiver: uuid
    receiver_type: str
    for_good_service: str
    amount: Decimal
    sender_balance_after: Decimal
    receiver_balance_after: Decimal

    def __init__(self,
                 sender: uuid,
                 sender_type: str,
                 receiver: uuid,
                 receiver_type: str,
                 for_good_service: str,
                 amount: Decimal,
                 sender_balance_after: Decimal,
                 receiver_balance_after: Decimal):
        self.id = uuid.uuid4()
        self.timestamp = datetime.datetime.now()
        self.sender = sender
        self.sender_type = sender_type
        self.receiver = receiver
        self.receiver_type = receiver_type
        self.for_good_service = for_good_service
        self.amount = amount
        self.sender_balance_after = sender_balance_after
        self.receiver_balance_after = receiver_balance_after

class Bank:
    accounts: list[BankAccount]
    transaction_log: list[TransactionLogEntry]

    def __init__(self):
        self.accounts = []
        self.transaction_log = []

    def create_account(self,
                       owner: Actor,
                       initial_balance: Decimal = Decimal(0)) -> BankAccount:
        """
        Create a new bank account for the given owner UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - BankAccount: The newly created bank account for the owner.
        """
        for account in self.accounts:
            if account.owner == owner.id:
                raise ValueError(f'Account already exists for owner {owner.name} (UUID: {owner.id})')
        account = BankAccount(initial_balance=initial_balance)
        account.owner = owner.id
        account.owner_type = type(owner).__name__
        self.accounts.append(account)
        return account
    

    def get_balance(self, owner: uuid) -> Decimal:
        """
        Get the balance of a bank account by the owner's UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - Decimal: The balance of the bank account associated with the given owner UUID.
        Raises:
            - ValueError: If no account is found for the given owner UUID.
        """
        account = self.get_account_by_owner(owner)
        return account.balance
    
    
    def get_account_by_owner(self, owner: uuid) -> BankAccount:
        """
        Get a bank account by the owner's UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - BankAccount: The bank account associated with the given owner UUID.
        Raises:
            - ValueError: If no account is found for the given owner UUID.
        """
        for account in self.accounts:
            if account.owner == owner:
                return account
        raise ValueError(f'No account found for owner {owner}')
    

    def transfer_money(self,
                       sender: uuid,
                       receiver: uuid,
                       amount: Decimal,
                       for_good_service: str) -> None:
        """
        Transfer money from one account to another.

        Args:
            - sender (uuid): The UUID of the sender's account.
            - receiver (uuid): The UUID of the receiver's account.
            - amount (Decimal): The amount of money to transfer.
            - for_good_service (str): One-word description of the good or service this transaction is for (e.g. "orange", "salary", etc.)

        Returns:
            None

        Raises:
            - ValueError: If either the sender or receiver account is not found.
            - NotEnoughMoneyError: If the sender does not have enough money to transfer.
        """
        sender_account = self.get_account_by_owner(sender)
        receiver_account = self.get_account_by_owner(receiver)
        if sender_account.balance < amount:
            raise NotEnoughMoneyError(f'Sender {sender} does not have enough money to transfer {amount}. Current balance: {sender_account.balance}')
        sender_account.balance -= amount
        receiver_account.balance += amount
        transaction_log_entry = TransactionLogEntry(
            sender=sender,
            sender_type=sender_account.owner_type,
            sender_balance_after=sender_account.balance,
            receiver=receiver,
            receiver_type=receiver_account.owner_type,
            receiver_balance_after=receiver_account.balance,
            for_good_service=for_good_service,
            amount=amount
        )
        self.transaction_log.append(transaction_log_entry)





class Retailer(Actor):
    def __init__(self, name: str, retailer_config: RetailerConfig, language_model_name: str | None = None,                langage_model_temperature: Decimal | None = None):
        super().__init__(name, language_model_name, langage_model_temperature)
        self.price = -1
        self.stock = 0
        self.competition_model = retailer_config.competition_model
        self.willing_to_sell = True if self.competition_model == CompetitionModel.BERTRAND else False
        self.notes_to_self = []
        self.reasoning = []
        self.last_bought_from_wholesaler = None
        self.last_sold_to_consumers = None
        self.last_profit = None

        if self.competition_model is CompetitionModel.COURNOT:
            raise NotImplementedError('Cournot competition model not implemented yet')
        

    def calculate_demand(self, wholesaler_stock: int, wholesaler_price: Decimal, my_money: Decimal) -> int:
        """
        Calculates the demand for goods from this Retailer based on the wholesaler's
        stock and price.

        The retailer demands as many goods as they can afford at the wholesaler's price.

        Args:
            - wholesaler_stock (int): The current stock available at the wholesaler.
            - wholesaler_price (Decimal): The current price per unit at the wholesaler.
        Returns:
            - int: The quantity of goods the retailer demands from the wholesaler.
        """

        max_affordable_quantity = int(my_money // wholesaler_price)
        demanded_quantity = min(max_affordable_quantity, wholesaler_stock)

        print(f'{self.name} can afford up to {max_affordable_quantity} units at the wholesaler price of {wholesaler_price}. (has {my_money} money)')

        economy_logger.info(f'{self.name} is requesting {demanded_quantity} goods from the wholesaler. (Max affordable: {max_affordable_quantity}, Wholesaler stock: {wholesaler_stock})')
        return demanded_quantity
    

    def calculate_willing_to_sell(self):
        """
        Determines if this Retailer is willing to sell goods to consumers based
        on the competition model, and the current wholesale price.

        Args:
            - wholesaler_price (Decimal): The current price per unit at the wholesaler.
        Returns:
            - bool: True if the retailer is willing to sell goods to consumers, False otherwise.
        """
        if self.competition_model == CompetitionModel.BERTRAND:
            # in Bertrand competition, retailers are always willing to sell to consumers
            # so long as they have stock, regardless of the wholesale price
            if self.stock <= 0:
                self.willing_to_sell = False
            else:
                self.willing_to_sell = True
        elif self.competition_model == CompetitionModel.COURNOT:
            raise NotImplementedError('Cournot competition model not implemented yet')
        else:
            raise ValueError(f'Unknown competition model: {self.competition_model}')
        return self.willing_to_sell

    
    def receive_goods(self, quantity):
        economy_logger.info(f'{self.name} received {quantity} units from the wholesaler...')
        self.stock += quantity
        return self.stock
    
    def set_price(self, price):
        economy_logger.info(f'{self.name} is setting price to {price}...')
        self.price = price

    def offer_goods(self):
        economy_logger.info(f'{self.name} is offering goods at price {self.price} each...')

    def process_sale(self):
        """
        Process the sale of a retailer's goods to a consumer. Reduce the retailer's
        stock by 1, and if the stock reaches 0, set the willing_to_sell flag to False.

        Returns:
            - None
        Raises:
            - ValueError: If the retailer cannot process the sale because stock is 0 or less
        """
        economy_logger.info(f'{self.name} is processing a sale...')
        if self.stock <= 0:
            raise ValueError(f'{self.name} cannot process sale because stock is {self.stock}')
        self.stock -= 1
        economy_logger.info(f'{self.name} has {self.stock} units left after the sale.')
        if self.stock <= 0:
            economy_logger.info(f'{self.name} has run out of stock and is no longer willing to sell to consumers.')
            self.willing_to_sell = False




class Wholesaler(Actor):
    price: Decimal
    stock: int
    is_unlimited_wholesaler: bool
    stocks: str

    def __init__(self, name, stocks: str = "Orange"):
        super().__init__(name)
        self.price = 0
        self.stock = 0
        self.is_unlimited_wholesaler = True
        self.stocks = stocks

    def set_price(self, price: Decimal) -> None:
        """
        Set the price for the wholesaler's goods.

        Args:
            - price (Decimal): The price to set for the wholesaler's goods.
        Raises:
            - ValueError: If the price is not a Decimal.
        """
        if not isinstance(price, Decimal):
            raise ValueError(f'Price must be a Decimal, got {type(price)}')
        economy_logger.info(f'{self.name} is setting price to {price}...')
        self.price = price

    def set_stock(self, stock: int):
        """
        Set the stock for the wholesaler's goods.

        Args:
            - stock (int): The stock to set for the wholesaler's goods.
        Raises:
            - ValueError: If the stock is not an integer.
        """
        if not isinstance(stock, int):
            raise ValueError(f'Stock must be an integer, got {type(stock)}')
        economy_logger.info(f'{self.name} is setting stock to {stock}...')
        self.stock = stock


    def get_current_price(self):
        """
        Get the current price of the wholesaler's goods.
        """
        return self.price

    def offer_goods(self):
        economy_logger.info(f'{self.name} is selling up to {self.stock} units at price {self.price} each...')
        return self.stock, self.price

    def process_sale(self):
        """
        Process a sale of exactly one of the wholesaler's goods.
        """
        economy_logger.info(f'{self.name} is processing a sale...')
        if self.is_unlimited_wholesaler:
            economy_logger.info(f'{self.name} is an unlimited wholesaler, so stock remains unchanged.')
        else:
            self.stock -= 1


class Consumer(Actor):
    earns_per_iteration: Decimal
    basket_count: int # the number of goods the consumer is holding
    total_utility: Decimal # the consumer's utility, which increases with each good they consume
    
    base_utility = Decimal('30.0')
    utility_decay: Decimal

    def __init__(self, name, consumer_config: ConsumerConfig, utility_decay: Decimal = Decimal('0.8')):
        super().__init__(name)
        self.earns_per_iteration = consumer_config.earns_per_iteration
        self.basket_count = 0
        self.total_utility = Decimal('0.0')
        self.utility_decay = utility_decay


    def marginal_utility(self, units_held: int) -> Decimal:
        """
        Utility gained from consuming the next good.

        This decays at a rate of <see cref="utility_decay"/> for each additional good consumed,
        starting from a base utility of <see cref="base_utility"/> for the first good.

        Args:
            - units_held (int): The number of units currently held by the consumer.
        Returns:
            - Decimal: The marginal utility of consuming the next good, based on the number of units currently held.
        """
        return self.base_utility * (self.utility_decay ** units_held)


    def consume_basket(self):
        economy_logger.info(f'{self.name} is consuming their basket of goods...')
        total_utility_this_iter = sum(self.marginal_utility(i) for i in range(self.basket_count))
        self.total_utility += total_utility_this_iter
        self.period_utility = total_utility_this_iter
        self.basket_count = 0
        economy_logger.info(f'{self.name} has consumed their basket and now has total utility of {self.total_utility}.')


    def willing_to_buy(self, price_point: Decimal) -> bool:
        """
        Check if the Consumer is willing to buy a good at the given price point
        based on their marginal utility. The consumer is willing to buy if the marginal utility
        of consuming the next good is greater than or equal to the price point.

        Args:
            - price_point (Decimal): The price at which the consumer is considering buying a good.
        Returns:
            - bool: True if the consumer is willing to buy at the given price point, False otherwise.     
        """
        return self.marginal_utility(self.basket_count) >= price_point

In [4]:
from clients.vlm_client import VLMClient
from configs.config import API_BASE_URL, API_KEY, PROVIDER
from utils.json_utils import clean_and_parse_json

model = "google/gemma-3n-e4b"

global_vlm_client = VLMClient(
    provider=PROVIDER,
    base_url=API_BASE_URL,
    api_key=API_KEY,
    model=model
)

api_system_prompt = """
You are a profit-maximising retailer in a competitive market.

Each week, you decide:
1) how many units to buy from the wholesaler (the wholesaler can always meet your demand).
2) what price to charge consumers. 

You'll be asked to respond with a quantity of goods to demand
from the wholesaler (provide as int), and a price to set for
those goods when selling to the consumers (provide as Decimal,
will be truncated to 2 decimal places). You should also give
a brief summary of your reasoning (1-2 short sentences) for 
why you are demanding that quantity, and setting that price. 
Finally, provide a brief note to yourself, which will be passed 
back to you next week (the note-to-self is strictly confidential
and will not be shared with anyone).

Respond only with valid JSON adhering to the following schema:
{
    demanded_quantity: int,
    retail_price: Decimal,
    reasoning: string,
    notes_to_self: string
}
"""

api_user_prompt = """
You are retailer "{retailer_name}". It's week {current_iteration}.

Your current situation:
- Your current stock: {current_stock}
- Your bank balance: {current_balance}
- Current wholesale price: {wholesaler_price} per unit
- Max affordable quantity at wholesale price: {max_affordable_quantity}

The current market situation:
- Number of consumers: {num_consumers}
- Number of competing retailers: {num_competing_retailers}

Market trading history:
- {market_history_md}

Your recent trading history:
- {trading_history_md}

Your recent note to self:
- {note_to_self}

Decide how many units to buy, and what retail price to set.
"""

def retailer_trading_history_to_markdown(retailer: Retailer, df_trading_history: DataFrame) -> str:
    """
    Convert the trading history of a retailer into a markdown string for inclusion in the prompt.

    Args:
        - retailer (Retailer): The retailer whose trading history we want to convert.
        - df_trading_history (DataFrame): The full trading history dataframe, containing all transactions for all retailers.
    Returns:
        - str: A markdown-formatted string summarising the retailer's trading history.
    """
    if df_trading_history.empty:
        return "No trading history yet."
    retailer_history = df_trading_history[df_trading_history['retailer_id'] == retailer.id]
    if retailer_history.empty:
        raise ValueError(f"No trading history found for retailer, even though df is not empty: retailer: {retailer.name} (ID: {retailer.id})")
    else:
        md = "| Week | Price | Qty Bought | Qty Sold | Stock | Balance | Profit |\n"
        md += "|------|-------|------------|----------|-------|---------|--------|\n"
        for index, row in retailer_history.iterrows():
            md += f"| {row['iteration']} | {row['price']} | {row['qty_bought']} | {row['qty_sold']} | {row['stock']} | {row['balance']} | {row['profit']} |\n"
        return md
    

def market_history_to_markdown(df_trading_history: DataFrame) -> str:
    """
    Convert the overall market trading history into a markdown string for inclusion in the prompt.

    Args:
        - df_trading_history (DataFrame): The full trading history dataframe, containing all transactions for all retailers.
    Returns:
        - str: A markdown-formatted string summarising the overall market trading history.
    """
    if df_trading_history.empty:
        return "No market trading history yet."
    else:
        md = "| Week | Retailer Name | Price |\n"
        md += "|------|---------------|-------|\n"
        for index, row in df_trading_history.iterrows():
            md += f"| {row['iteration']} | {row['retailer_name']} | {row['price']} |\n"
        return md


def query_language_model_for_retailer_strategy(retailer: Retailer,
                                               wholesaler_price: Decimal,
                                               current_balance: Decimal,
                                               df_trading_history: pd.DataFrame,
                                               current_iteration: int,
                                               num_consumers: int,
                                               num_competing_retailers: int) -> (int, Decimal, str, str):
    try:
        query_temperature = retailer.language_model_temperature if retailer.language_model_temperature is not None else global_vlm_client.temperature
        response = global_vlm_client.query(
            system_prompt=api_system_prompt,
            prompt=api_user_prompt.format(
                retailer_name=retailer.name,
                current_iteration=current_iteration,
                current_stock=retailer.stock,
                current_balance=current_balance,
                wholesaler_price=wholesaler_price,
                max_affordable_quantity=int(current_balance // wholesaler_price),
                num_consumers=num_consumers,
                num_competing_retailers=num_competing_retailers,
                note_to_self=retailer.notes_to_self[-1] if retailer.notes_to_self else 'No note set',
                trading_history_md=retailer_trading_history_to_markdown(retailer, df_trading_history),
                market_history_md=market_history_to_markdown(df_trading_history)
            ),
            language_model_path=retailer.language_model_name or global_vlm_client.model,
            temperature=float(query_temperature)
        )
        print(f'Actor response from language model: {response}')
        response_json = clean_and_parse_json(response)
        retail_price = Decimal(response_json['retail_price']).quantize(Decimal('0.01'))
        return response_json['demanded_quantity'], \
                            retail_price, \
                            response_json['reasoning'], \
                            response_json['notes_to_self']
    except Exception as e:
        economy_logger.error(f"Error querying language model for retailer strategy: {e}")
        raise e

## The Economy

This cell contains the Economy class. That's the schema for the program, and contains the different types of Actors in the economy, and handles the way they interact with each other.

In [5]:
class EconomyReport:
    retailer_history: DataFrame
    consumer_history: DataFrame
    def __init__(self):
        self.retailer_history = pd.DataFrame()
        self.consumer_history = pd.DataFrame()


class Economy:
    wholesaler: Wholesaler = None
    retailers: list[Retailer] = None
    consumers: list[Consumer] = None
    salary_processor: Actor = None
    bank: Bank

    def __init__(self, 
                 wholesaler_config: WholesalerConfig,
                 retailer_config: RetailerConfig,
                 consumer_config: ConsumerConfig):
        self.bank = Bank()
        self.wholesaler = self.setup_wholesaler(wholesaler_config)
        self.retailers = self.setup_retailers(retailer_config)
        self.consumers = self.setup_consumers(consumer_config)
        self.salary_processor = self.setup_salary_processor()
        self.economy_report = EconomyReport()


    def pay_salary(self, consumer: Consumer, employer = None):
        """
        Pay a salary to a consumer from the specified employer. If no employer is specified, 
        the salary is paid by the salary processor.

        Args:
            - consumer (Consumer): The consumer receiving the salary.
            - employer (Actor, optional): The employer paying the salary. Defaults to the salary processor.
        """
        if not isinstance(consumer, Consumer):
            raise ValueError(f'Expected consumer recipient of salary to be an instance of Consumer, got {type(consumer)}')
        if employer is None:
            employer = self.salary_processor

        economy_logger.info(f'Paying salary of {consumer.earns_per_iteration} to {consumer.name} from employer {employer.name}...')
        self.bank.transfer_money(employer.id, consumer.id, consumer.earns_per_iteration, for_good_service='salary')
        

    def process_wholesaler_transaction(self, wholesaler: Wholesaler,retailer: Retailer, quantity: int) -> int:
        """
        Process a transaction between a retailer and the wholesaler, including money transfer and inventory updates.

        Args:
            - wholesaler (Wholesaler): The wholesaler involved in the transaction.
            - retailer (Retailer): The retailer involved in the transaction.
            - quantity (int): The quantity of goods being purchased by the retailer from the wholesaler.
        Returns:
            - int: The quantity of goods that were successfully purchased by the retailer from the wholesaler
        """
        total_cost = wholesaler.get_current_price() * quantity
        print(f'{retailer.name} is attempting to buy {quantity} units from {wholesaler.name} for a total cost of {total_cost}...')
        if total_cost > self.bank.get_balance(retailer.id):
            economy_logger.error(f'{retailer.name} cannot afford to buy requested {quantity} units from {wholesaler.name} at total cost {total_cost}. Current balance: {self.bank.get_balance(retailer.id)}. Purchasing the maximum affordable quantity instead...')
            return self.process_wholesaler_transaction(wholesaler, retailer, int(self.bank.get_balance(retailer.id) // wholesaler.get_current_price()))
        try:
            self.bank.transfer_money(retailer.id, wholesaler.id, total_cost, for_good_service=f'wholesale_{wholesaler.stocks}_x_{quantity}')
            for _ in range(quantity):
                wholesaler.process_sale()
            retailer.receive_goods(quantity)
        except NotEnoughMoneyError as e:
            raise NotEnoughMoneyError(f'{retailer.name} does not have enough money to buy {quantity} units from {wholesaler.name} at total cost {total_cost}. Current balance: {self.bank.get_balance(retailer.id)}')
        return quantity


    def process_retailer_transaction(self, retailer: Retailer, consumer: Consumer, good_or_service: str):
        """
        Process a transaction between a retailer and a consumer, including money transfer and inventory updates.

        Args:
            - retailer (Retailer): The retailer involved in the transaction.
            - consumer (Consumer): The consumer involved in the transaction.
            - good_or_service (str): A one-word description of the good or service being sold (e.g. "orange", "apple", "banana", etc.)
        """
        total_cost = retailer.price
        print(f'{consumer.name} is buying 1 unit from {retailer.name} for a total cost of {total_cost}...')
        try:
            self.bank.transfer_money(consumer.id, retailer.id, total_cost, for_good_service=f'retail_{good_or_service}_x_1')
            retailer.process_sale()
            consumer.basket_count += 1
            retailer.last_sold_to_consumers += 1
        except NotEnoughMoneyError as e:
            economy_logger.error(f'Transaction failed due to low funds: {e}')


    def run_loop(self, current_iteration: int, init_time: datetime) -> DataFrame:
        economy_logger.info(f'Running economy loop for iteration {current_iteration}...')
        
        # wholesaler sets prices and offers goods

        print(f'--- Wholesaler Loop Iteration {current_iteration} ---')
        self.wholesaler.set_price(Decimal('10.0'))
        self.wholesaler.set_stock(1_000_000)
        wholesaler_stock, wholesaler_price = self.wholesaler.offer_goods()

        print(f'--- Retailer Loop Iteration {current_iteration} ---')

        # a DUMB retailer picks as many goods as they can afford at the wholesaler price, and sets
        # their price to around 1.5x the wholesale price to add some variation to the pricing strategy
        # a SMART retailer queries a language model with information about the market. The language model
        # returns a recommended quantity to buy from the wholesaler, and a recommended price to set
        # for the consumers in the market. The retailer always obeys the language model recommendation.
        run_lang_model_retailer = True

        for retailer in self.retailers:
            print(f'{retailer.name} has {retailer.stock} units in stock before the iteration')
            
            retailer.last_bought_from_wholesaler = 0
            retailer.last_sold_to_consumers = 0
            
            if not run_lang_model_retailer:
                demanded_quantity = retailer.calculate_demand(wholesaler_stock, wholesaler_price, self.bank.get_balance(retailer.id))
                # set price to a random figure between 1.4 and 1.6 times the wholesale price to add some variation to the pricing strategy      
                price_randomiser = random.uniform(1.4, 1.6)
                retailer.set_price(wholesaler_price * Decimal(price_randomiser))
            else: 
                # query the language model for the demanded quantity and price to set, then set those values for the retailer
                demanded_quantity, intended_price, reasoning, notes_to_self = \
                    query_language_model_for_retailer_strategy(retailer,
                                                               wholesaler_price,
                                                               self.bank.get_balance(retailer.id),
                                                                df_trading_history=self.economy_report.retailer_history,
                                                                current_iteration=current_iteration,
                                                                num_consumers=len(self.consumers),
                                                                num_competing_retailers=len(self.retailers)-1
                                                               )
                retailer.set_price(intended_price)
                demanded_quantity = demanded_quantity
                retailer.notes_to_self.append(notes_to_self)
                retailer.reasoning.append(reasoning)

            processed_purchase_quantity = self.process_wholesaler_transaction(self.wholesaler, retailer, demanded_quantity)
            retailer.last_bought_from_wholesaler = processed_purchase_quantity
            retailer.calculate_willing_to_sell()
            print(f'{retailer.name} has {retailer.stock} units in stock after the iteration setup, and is {"willing" if retailer.willing_to_sell else "not willing"} to sell.')



        print(f'--- Salary Payment Loop (employer of last resort) Iteration {current_iteration} ---')
        for consumer in self.consumers:
            print(f'{consumer.name} earns {consumer.earns_per_iteration} money at the start of the iteration...')
            self.pay_salary(consumer)

        # --- Consumer Loop ---
        print(f'--- Consumer Loop Iteration {current_iteration} ---')
        self.retailers.sort(key=lambda r: r.price)

        # Round-robin: each consumer buys 1 unit per round, loop until termination
        active_consumers = list(self.consumers)  # consumers still able to buy

        while active_consumers:
            random.shuffle(active_consumers)
            consumers_to_remove = []
            
            # Check global termination: no retailers willing to sell
            willing_retailers = [r for r in self.retailers if r.willing_to_sell]
            if not willing_retailers:
                print('No retailers willing to sell. Ending consumer loop.')
                break
            
            for consumer in active_consumers:
                # Find cheapest willing retailer this consumer can afford
                retailer_to_buy_from = None
                for retailer in willing_retailers:
                    if (self.bank.get_balance(consumer.id) >= retailer.price and consumer.willing_to_buy(retailer.price)):
                        retailer_to_buy_from = retailer
                        break
                
                if retailer_to_buy_from is None:
                    consumers_to_remove.append(consumer)
                    continue
                
                # Buy exactly 1 unit (one turn)
                try:
                    self.process_retailer_transaction(retailer_to_buy_from, consumer, good_or_service="Orange")
                except NotEnoughMoneyError:
                    consumers_to_remove.append(consumer)
                    continue
                
                # Refresh willing retailers after the sale
                willing_retailers = [r for r in self.retailers if r.willing_to_sell]
                if not willing_retailers:
                    break
            
            # Remove consumers who can no longer buy
            for c in consumers_to_remove:
                active_consumers.remove(c)
            
        for consumer in self.consumers:
            consumer.consume_basket() # consume the basket at the end of the iteration
            print(f'{consumer.name} has {self.bank.get_balance(consumer.id)} money, and {consumer.total_utility} utility after the iteration')

        for retailer in self.retailers:
            # calculate profit and store to retailer.profit
            revenue = retailer.last_sold_to_consumers * retailer.price
            cost = retailer.last_bought_from_wholesaler * self.wholesaler.get_current_price()
            retailer.last_profit = revenue - cost
            print(f'{retailer.name} made a profit of {retailer.last_profit} in this iteration (Revenue: {revenue}, Cost: {cost})')


        consumer_rows = [
            {
                'iteration': current_iteration,
                'consumer_id': consumer.id,
                'consumer_name': consumer.name,
                'balance': self.bank.get_balance(consumer.id),
                'total_utility': consumer.total_utility,
                'basket_count': consumer.basket_count,
                'period_utility': consumer.period_utility
            }
            for consumer in self.consumers
        ]
        self.economy_report.consumer_history = pd.concat(
            [self.economy_report.consumer_history, pd.DataFrame(consumer_rows)],
            ignore_index=True
        )

        retailer_rows = [
            {
                'iteration': current_iteration,
                'retailer_id': retailer.id,
                'retailer_name': retailer.name,
                'retailer_model': retailer.language_model_name or global_vlm_client.model,
                'retailer_temperature': retailer.language_model_temperature,
                'balance': self.bank.get_balance(retailer.id),
                'stock': retailer.stock,
                'price': retailer.price,
                'qty_bought': retailer.last_bought_from_wholesaler,
                'qty_sold': retailer.last_sold_to_consumers,
                'profit': retailer.last_profit,
                'reasoning': retailer.reasoning[-1] if retailer.reasoning else '',
                'notes_to_self': retailer.notes_to_self[-1] if retailer.notes_to_self else ''

            } for retailer in self.retailers
        ]
        self.economy_report.retailer_history = pd.concat(
            [self.economy_report.retailer_history, pd.DataFrame(retailer_rows)],
            ignore_index=True
        )

        self.iteration_report()
        transaction_df = self.transaction_report()
        save_df_to_csv = True
        if save_df_to_csv:
            csv_path = log_path / f'{init_time.strftime("%H%M%S")}_transactions_iteration_{current_iteration}.csv'
            transaction_df.to_csv(csv_path, index=False)
            economy_logger.info(f'Transaction report for iteration {current_iteration} saved to {csv_path}')

        economy_logger.info(f'Economy loop for iteration {current_iteration} complete.\n\n')
        return transaction_df


    def setup_wholesaler(self, wholesaler_config: WholesalerConfig):
        wholesaler = Wholesaler(wholesaler_config.name)
        self.bank.create_account(wholesaler, initial_balance=Decimal())
        return wholesaler

    def setup_consumers(self, consumer_config: ConsumerConfig):
        """
        Setup the consumers in the economy based on the provided configuration.

        Each consumer has an earning rate, defined in ConsumerConfig, which determines
        how much money they earn at the start of each iteration.

        Args:
            - consumer_config (ConsumerConfig): The configuration for setting up consumers
        Returns:
            - list[Consumer]: A list of Consumer instances set up according to the configuration.
        """
        def setup_consumer(name, consumer_config, utility_decay=Decimal('0.8')):
            return Consumer(name, consumer_config, utility_decay)
        
        consumers = []
        economy_logger.debug(f'Setting up {consumer_config.num_consumers} consumers...')
        for i in range(consumer_config.num_consumers):
            consumer = setup_consumer(f'Consumer {i+1}', consumer_config)
            consumers.append(consumer)
            self.bank.create_account(consumer, initial_balance=Decimal(consumer_config.earns_per_iteration))
        return consumers


    def setup_retailers(self, retailer_config: RetailerConfig):
        def setup_retailer(name, language_model, language_model_temperature):
            return Retailer(name, retailer_config, language_model, language_model_temperature)
        
        retailers = []
        
        economy_logger.debug(f'Setting up {retailer_config.num_retailers} retailers...')

        for i in range(retailer_config.num_retailers):
            lang_model_conf = available_lang_model_and_temperature_combinations[i % len(available_lang_model_and_temperature_combinations)]
            retailer_name = available_retailer_names[i % len(available_retailer_names)]
            
            retailer = setup_retailer(retailer_name,
                                      lang_model_conf['model'],
                                      lang_model_conf['temperature'])
            self.bank.create_account(retailer, initial_balance=Decimal(retailer_config.starting_money))
            retailers.append(retailer)
        return retailers


    def setup_salary_processor(self):
        """
        Set up the salary processor, which is an abstract Actor which has loads of cash.
        This is used to pay consumers their earnings at the start of each iteration.

        In a more complex economy simulation, salaries would be paid by the retailers and wholesalers
        (employers) based on the work done by the consumers. But for simplicity in this model, 
        there's a single Actor with "unlimited-ish" money who pays out earnings.

        Returns:
            - Actor: The salary processor Actor instance.
        """
        salary_processor = Actor('Salary Processor')
        self.bank.create_account(salary_processor, initial_balance=Decimal(1_000_000_000_000))
        return salary_processor

    def iteration_report(self):
        print('--- Iteration Report ---')

        print(f'Salary Processor: {self.salary_processor.name}, Money: {self.bank.get_account_by_owner(self.salary_processor.id).balance}')
        print(f'Wholesaler: {self.wholesaler.name}, Money: {self.bank.get_account_by_owner(self.wholesaler.id).balance}, Stock: {self.wholesaler.stock}, Price: {self.wholesaler.price}')
        for retailer in self.retailers:
            print(f'Retailer: {retailer.name}, Money: {self.bank.get_account_by_owner(retailer.id).balance}, Stock: {retailer.stock}, Price: {retailer.price}')
        for consumer in self.consumers:
            print(f'Consumer: {consumer.name}, Money: {self.bank.get_balance(consumer.id)}, Utility: {consumer.total_utility}, Basket Count: {consumer.basket_count}')
        print('--- End of Report ---')

    def transaction_report(self) -> DataFrame:
        print('--- Transaction Report ---')
        transaction_data = [{
            'timestamp': entry.timestamp,
            'sender': entry.sender,
            'sender_type': entry.sender_type,
            'sender_balance_after': entry.sender_balance_after,
            'receiver': entry.receiver,
            'receiver_type': entry.receiver_type,
            'receiver_balance_after': entry.receiver_balance_after,
            'for_good_service': entry.for_good_service,
            'amount': entry.amount
        } for entry in self.bank.transaction_log]
        df = DataFrame(transaction_data)
        print(df)
        print('--- End of Report ---')
        return df

In [6]:
def save_economy_report(economy: Economy, filename_prefix: str):
    """
    Saves the economy history reports for consumers and retailers
    to csv files with the given filename prefix and a timestamp.

    Args:
        - economy (Economy): The Economy instance containing the reports to save.
        - filename_prefix (str): The prefix to use for the generated CSV filenames.
    Returns:
        - None
    """
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    consumer_report_path = log_path / f'{filename_prefix}_consumer_history_{timestamp}.csv'
    retailer_report_path = log_path / f'{filename_prefix}_retailer_history_{timestamp}.csv'
    economy.economy_report.consumer_history.to_csv(consumer_report_path, index=False)
    economy.economy_report.retailer_history.to_csv(retailer_report_path, index=False)
    print(f'Consumer history saved to {consumer_report_path}')
    print(f'Retailer history saved to {retailer_report_path}')

## Application Loop

This cell runs the application in a loop.

In [7]:
from utils.plotly_graphs.consumer_graphs import plot_consumer_period_utility_vs_balance, plot_consumer_total_utility_vs_balance
from utils.plotly_graphs.money_graphs import plot_retailer_money, plot_retailer_profit
from utils.plotly_graphs.price_graphs import plot_price_band
from IPython.display import display, Markdown

def render_graphs(economy_sim: Economy):
    plot_consumer_period_utility_vs_balance(economy_sim.economy_report.consumer_history).show()
    plot_consumer_total_utility_vs_balance(economy_sim.economy_report.consumer_history).show()
    
    plot_price_band(economy_sim.economy_report.retailer_history).show()
    plot_retailer_profit(economy_sim.economy_report.retailer_history).show()
    plot_retailer_money(economy_sim.economy_report.retailer_history).show()

def write_iteration_report(economy_sim: Economy):
    last_iteration = economy_sim.economy_report.retailer_history["iteration"].iloc[-1]
    last_iteration_retailer_history = economy_sim.economy_report.retailer_history[economy_sim.economy_report.retailer_history["iteration"] == last_iteration]
    md = f'## Iteration Summary for Iteration {last_iteration}\n\n'
    md += '### Retailer Summary\n\n'
    md += '| Retailer / model | Balance | Stock | Px | Qty Bought / Sold | Profit | Reasoning | Notes to Self |\n'
    md += '|----------|---------|-------|-------|------------|--------|-----------|-----------|\n'
    for index, row in last_iteration_retailer_history.iterrows():
        md += f"| {row['retailer_name']} / {row['retailer_model']} ({row['retailer_temperature']}) | {row['balance']} | {row['stock']} | {row['price']} | {row['qty_bought']} / {row['qty_sold']} | {row['profit']} | {row['reasoning']} | {row['notes_to_self']} |\n"
    display(Markdown(md))


In [ ]:
from IPython.display import clear_output
import ipywidgets as widgets


def main(iterations):
    wholesaler_config = WholesalerConfig()
    
    run_competition_model = CompetitionModel.BERTRAND
    retailer_config = RetailerConfig()
    retailer_config.competition_model = run_competition_model
    retailer_config.num_retailers = 4

    consumer_config = ConsumerConfig()
    consumer_config.num_consumers = 55
    consumer_config.earns_per_iteration = Decimal('22.72')
    
    economy = Economy(wholesaler_config, retailer_config, consumer_config)

    print(f'Starting {run_competition_model.name} economy simulation for {iterations} iterations...')
    
    init_time = datetime.datetime.now()
    for i in range(iterations):
        loop_init_time = datetime.datetime.now()
        print(f'Running iteration {i + 1}...')
        economy.run_loop(i + 1, init_time)
        clear_output(wait=True)
        print(f'Completed iteration {i + 1}/{iterations} in {(datetime.datetime.now() - loop_init_time).total_seconds():.2f} seconds.')
        render_graphs(economy)
        write_iteration_report(economy)

    save_economy_report(economy, filename_prefix='bertrand_competition')
    print('Economy simulation complete after {} iterations.'.format(iterations))

    return economy


economy_sim = main(25)

2026-06-02 11:11:59,338 - economy - INFO - Completed iteration 1/25 in 63.18 seconds.


## Iteration Summary for Iteration 1

### Retailer Summary

| Retailer / model | Balance | Stock | Px | Qty Bought / Sold | Profit | Reasoning | Notes to Self |
|----------|---------|-------|-------|------------|--------|-----------|-----------|
| Sally's Sales / mistralai/ministral-3-3b (0.3) | 319.80 | 0 | 15.99 | 20 / 20 | 119.80 | Maximize profit by purchasing the maximum affordable quantity (20 units) at wholesale cost ($10 each), then setting a premium retail price ($15.99) to capture consumer demand in a competitive market with few competitors. | Start with aggressive pricing to establish market position; monitor competitor reactions next week. |
| Eve's Essentials / qwen/qwen3.5-9b (0.7) | 290.00 | 0 | 17.50 | 12 / 12 | 90.00 | Starting with 12 units (60% of affordable max) balances risk and market entry while setting price at 17.50 to capture margin above the 10.0 wholesale cost. This conservative approach allows me to test demand without overcommitting capital in an unknown market. | Week 1 - need to learn consumer willingness to pay. Monitor sales velocity and competitor responses before expanding inventory next week. |
| Charlie's Exquisite Goods / qwen/qwen3.5-9b (0.3) | 290.00 | 0 | 17.50 | 12 / 12 | 90.00 | Starting with a moderate inventory level (12 units) to test market demand without overextending my bank balance. Setting price at 17.50 provides healthy margin while remaining competitive against the 5 other retailers. | Monitor sales velocity and competitor pricing next week. If consumers respond well, consider increasing quantity or testing higher prices. Need to learn the demand curve early. |
| Alice's Emporium / google/gemma-3-4b (0.3) | 350.00 | 0 | 20.00 | 15 / 15 | 150.00 | Given the competition and limited stock, I'll buy 15 units to ensure sufficient supply while maintaining a price of $20.00 to capture some profit margin. | Monitor competitor pricing closely and consider increasing quantity next week if demand remains high. |
| Tom's Treasures / google/gemma-3-4b (0.7) | 395.00 | 0 | 23.00 | 15 / 15 | 195.00 | Given the competitive market with multiple retailers and a limited budget, I'll buy 15 units to ensure sufficient stock while keeping my price slightly above the wholesale cost, aiming for a healthy profit margin. | Monitor competitor pricing closely and consider increasing quantity next week if demand remains high. |
| Bobs Bargains / mistralai/ministral-3-3b (0.7) | 100.0 | 10 | 25.99 | 10 / 0 | -100.00 | Demand 10 units to establish a moderate lead in the market while controlling costs. Price at **$26.99**—below competitors' likely average ($28–$30) to capture demand early and maximize profits through premium positioning. | **Market Entry Play**: Secure initial volume before rivals; observe competitor reactions next week. Monitor price elasticity for adjustments if demand spikes. |


2026-06-02 11:11:59,704 - economy - INFO - Running iteration 2...
2026-06-02 11:11:59,705 - economy - INFO - Running economy loop for iteration 2...
2026-06-02 11:11:59,705 - economy - INFO - --- Wholesaler Loop Iteration 2 ---
2026-06-02 11:11:59,706 - economy - INFO - Orange Wholesaler is setting price to 10.0...
2026-06-02 11:11:59,706 - economy - INFO - Orange Wholesaler is setting stock to 1000000...
2026-06-02 11:11:59,706 - economy - INFO - Orange Wholesaler is selling up to 1000000 units at price 10.0 each...
2026-06-02 11:11:59,707 - economy - INFO - --- Retailer Loop Iteration 2 ---
2026-06-02 11:11:59,707 - economy - INFO - Sally's Sales has 0 units in stock before the iteration
2026-06-02 11:12:02,487 - economy - INFO - Actor response from language model: ```json
{
    "demanded_quantity": 25,
    "retail_price": 14.99,
    "reasoning": "Demand more inventory to capitalize on the competitive pricing advantage from Week 1 while maintaining a premium position below Eve’s/Char

ReadTimeout: HTTPConnectionPool(host='localhost', port=1234): Read timed out. (read timeout=120)